# SumExplicitStack (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/SumExplicitStack_cpp.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `SumExplicitStack`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/SumExplicitStack_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/SumExplicitStack_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `FibRec` (PascalCase).

## From recursion to an explicit stack: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the sum $a[1]+\dots+a[n]$. The iterative version of the halving sum: the stack $P$ contains the pairs $(l,r)$ of the sub-arrays still to be summed and does the job of the call stack. The stack never contains more than $\Theta(\log_2 n)$ pairs.

On the array $a=[4,7,1,9]$ of the chapter we print the trace (with indices from 0: the pair $(1,4)$ of the chapter is here $(0,3)$; in each row the top of the stack is on the right) and we compare with `SumIterative`, `sumRecHalving` and `sumRecDecrease` of the code of the course, with the maximum depth of the recursion. Then we print the call stack of `sumRecDecrease(a, 3)` on $a=[4,7,1]$ (example of the chapter): the additions are performed only *after* the base case, while the stack is emptied, and at some point the active calls are $n+1$.

In [1]:
%%writefile explicit_stack.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int maxP = 0;   // maximum size of the stack P

typedef pair<int, int> Pair;

// SumExplicitStack(a, n): sum of a[0..n-1] with an explicit stack
long long SumExplicitStack(int *a, int n)
{
    long long s = 0;
    vector<Pair> P;                 // stack of the sub-arrays to be summed
    P.push_back({0, n - 1});
    maxP = max(maxP, (int)P.size());
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];           // base case
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            maxP = max(maxP, (int)P.size());
        }
    }
    return s;
}

string FmtP(const vector<Pair> &P)
{
    string s = "[";
    for (size_t i = 0; i < P.size(); i++)
    {
        if (i > 0) s += ",";
        s += "(" + to_string(P[i].first) + "," + to_string(P[i].second) + ")";
    }
    return s + "]";
}

long long SumExplicitStackTrace(int *a, int n)
{
    long long s = 0;
    vector<Pair> P = {{0, n - 1}};
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];
            printf("pop (%d,%d): base case, s=%lld => P=%s\n", l, r, s, FmtP(P).c_str());
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            printf("pop (%d,%d): mid=%d, push (%d,%d) and (%d,%d) => P=%s, s=%lld\n", l, r, mid, mid + 1, r, l, mid, FmtP(P).c_str(), s);
        }
    }
    return s;
}

// ---- the other sums of the course code, with the depth ----
int depth = 0;
int maxDepth = 0;

long long sumIt(int *a, int n)
{
    long long s = 0;
    for (int i = 0; i < n; i++)
    {
        s = s + a[i];
    }
    return s;
}

long long sumRecHalving(int *a, int l, int r)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (l == r)
    {
        result = a[l];
    }
    else
    {
        int mid = l + (r - l) / 2;
        result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
    }
    depth--;
    return result;
}

long long sumRecDecrease(int *a, int n)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (n == 0)
    {
        result = 0;
    }
    else
    {
        result = sumRecDecrease(a, n - 1) + a[n - 1];
    }
    depth--;
    return result;
}

// ---- the call stack of sumRecDecrease(a, 3) ----
int stepNo = 0;
int maxActive = 0;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

void Emit(const vector<string> &stack, const string &action)
{
    stepNo++;
    if ((int)stack.size() > maxActive) maxActive = stack.size();
    string st;
    for (size_t i = 0; i < stack.size(); i++)
    {
        if (i > 0) st += " ";
        st += stack[i];
    }
    printf("%2d | %s | %s\n", stepNo, st.c_str(), action.c_str());
}

long long SumRecDecreaseTrace(int *a, int n, vector<string> &stack)
{
    stack.push_back("s(" + to_string(n) + ")");
    if (n == 0)
    {
        Emit(stack, "n=0: return 0");
        stack.pop_back();
        return 0;
    }
    Emit(stack, "n=" + to_string(n) + " != 0: call s(" + to_string(n - 1) + "), then add a[" + to_string(n - 1) + "]");
    long long r = SumRecDecreaseTrace(a, n - 1, stack);
    Emit(stack, "return " + to_string(r) + "+a[" + to_string(n - 1) + "]=" + to_string(r + a[n - 1]));
    stack.pop_back();
    return r + a[n - 1];
}

#ifndef NOMAIN
int main()
{
    int a[] = {4, 7, 1, 9};
    int n = 4;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("SumExplicitStack(a, 4):\n");
    long long res = SumExplicitStackTrace(a, n);
    maxP = 0;
    SumExplicitStack(a, n);
    printf("result %lld, maximum size of P = %d\n", res, maxP);
    printf("\n");
    printf("sumIt                = %lld\n", sumIt(a, n));
    maxDepth = 0;
    long long r1 = sumRecHalving(a, 0, n - 1);
    printf("sumRecHalving        = %lld   maximum depth = %d\n", r1, maxDepth);
    maxDepth = 0;
    long long r2 = sumRecDecrease(a, n);
    printf("sumRecDecrease       = %lld   maximum depth = %d\n", r2, maxDepth);

    int b[] = {4, 7, 1};
    printf("\n");
    printf("a = %s   sumRecDecrease(a, 3)\n", Fmt(b, 3).c_str());
    printf(" # | stack of the active calls | action\n");
    vector<string> stack;
    res = SumRecDecreaseTrace(b, 3, stack);
    printf("result %lld, active calls at most = %d = n+1\n", res, maxActive);
    return 0;
}
#endif

Writing explicit_stack.cpp


In [2]:
!g++ -std=c++17 -O2 -o explicit_stack explicit_stack.cpp && ./explicit_stack

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) and (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) and (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): base case, s=4 => P=[(2,3),(1,1)]
pop (1,1): base case, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) and (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): base case, s=12 => P=[(3,3)]
pop (3,3): base case, s=21 => P=[]
result 21, maximum size of P = 3

sumIt                = 21
sumRecHalving        = 21   maximum depth = 3
sumRecDecrease       = 21   maximum depth = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | stack of the active calls | action
 1 | s(3) | n=3 != 0: call s(2), then add a[2]
 2 | s(3) s(2) | n=2 != 0: call s(1), then add a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: call s(0), then add a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
result 12, active calls at most = 4 = n+1


**Try it yourself.** Change the array: the explicit stack never contains more than $\lceil\log_2 n\rceil+1$ pairs.

In [3]:
%%writefile prova_explicit_stack.cpp
#define NOMAIN
#include "explicit_stack.cpp"

int main()
{
    // Change the array: the explicit stack never contains more than ceil(log2 n) + 1 pairs.
    vector<int> c(100);
    for (int i = 0; i < 100; i++) c[i] = i + 1;
    maxP = 0;
    long long s = SumExplicitStack(c.data(), 100);
    printf("%lld %d\n", s, maxP);
    return 0;
}

Writing prova_explicit_stack.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_explicit_stack prova_explicit_stack.cpp && ./prova_explicit_stack

5050 8


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).